In [ ]:
import pandas as pd
import numpy as np
from sklearn.manifold import TSNE
from sklearn.decomposition import PCA
from sklearn.preprocessing import OneHotEncoder
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import plotly.express as px

In [ ]:
df_vae_trained = pd.read_csv('data/oxt_1.csv')
df_vae_trained['label'] = 1

In [ ]:
df_vae_result = pd.read_csv('result_vae_80.csv')
df_vae_result['label'] = 0

In [ ]:
df = pd.concat([df_vae_result[['sequence', 'label']],df_vae_trained[['sequence', 'label','dff']]])
df.fillna(-1, inplace=True)
df = df.reset_index(drop=True)
df

In [ ]:
def one_hot_encode(seq):
    mapping = {'A': [1, 0, 0, 0], 'C': [0, 1, 0, 0], 'G': [0, 0, 1, 0], 'T': [0, 0, 0, 1]}
    return [mapping[i] for i in seq]

df['sequence_onehot'] = df['sequence'].apply(one_hot_encode)

sequences_array = np.array(df['sequence_onehot'].tolist())

sequences_array = sequences_array.reshape(sequences_array.shape[0], -1)

pca = PCA(n_components=2)  
sequences_pca = pca.fit_transform(sequences_array)

df['PCA 1'] = sequences_pca[:,0]
df['PCA 2'] = sequences_pca[:,1]

tsne = TSNE(n_components=2)
sequences_tsne = tsne.fit_transform(sequences_array)

df['t-SNE 1'] = sequences_tsne[:,0]
df['t-SNE 2'] = sequences_tsne[:,1]

In [ ]:
df.drop('sequence_onehot',axis=1,inplace=True)
df.to_csv('plotty.csv',index=False)

In [ ]:
colors = df['label'].map({0: 'red', 1: 'blue'})

plt.figure(figsize=(10, 8), dpi=300)
plt.scatter(df['PCA 1'], df['PCA 2'], c=colors)

red_patch = mpatches.Patch(color='red', label='vae')
blue_patch = mpatches.Patch(color='blue', label='DF/F')
plt.legend(handles=[red_patch, blue_patch])

plt.xlabel('PCA 1')
plt.ylabel('PCA 2')
plt.savefig('pca.png')
plt.show()

In [ ]:
colors = df['label'].map({0: 'red', 1: 'blue'})

plt.figure(figsize=(10, 8), dpi=300)
plt.scatter(df['t-SNE 1'], df['t-SNE 2'], c=colors)

red_patch = mpatches.Patch(color='red', label='vae')
blue_patch = mpatches.Patch(color='blue', label='DF/F')
plt.legend(handles=[red_patch, blue_patch])


plt.xlabel('t-SNE 1')
plt.ylabel('t-SNE 2')
plt.savefig('tsne.png')
plt.show()

In [ ]:
from sklearn.cluster import KMeans
import matplotlib.pyplot as plt
x,y ='t-SNE 1','t-SNE 2'
k = 3
kmeans_model = KMeans(n_clusters=k, n_init=10).fit(df[[x, y]])
labels = kmeans_model.labels_
df['cluster_kmeans_tnse'] = labels
plt.figure(figsize=(10, 8), dpi=300)
plt.scatter(df[x], df[y], c=labels, cmap='jet')
plt.title('K-means')
plt.xlabel(x)
plt.ylabel(y)
plt.savefig('K-means-tsne.png')

In [ ]:
from sklearn.mixture import GaussianMixture

gmm = GaussianMixture(n_components=3) 

gmm.fit(df[['t-SNE 1', 't-SNE 2']])
labels = gmm.predict(df[['t-SNE 1', 't-SNE 2']])

df['cluster_gm_tsne'] = labels

plt.figure(figsize=(10, 8), dpi=300)
plt.scatter(df['t-SNE 1'], df['t-SNE 2'], c=labels, cmap='jet')
plt.title('Gaussian Mixture')
plt.xlabel('t-SNE 1')
plt.ylabel('t-SNE 2')
plt.savefig('gm-tsne.png')

In [ ]:
df